# ☸️ Phase 1: Installation + Cluster Setup

Install tools → Create cluster → Understand the architecture

**What you'll do:**
1. Install `kubectl`, `kind`, Docker
2. (Optional) Install `k9s`
3. Create & verify a local K8s cluster
4. Learn the architecture

## Tool Overview

| Tool | What it does |
|------|-------------|
| **kubectl** | CLI that talks to the K8s API server |
| **kind** | K8s clusters inside Docker containers |
| **Docker** | Container runtime — kind needs it |
| **k9s** | Terminal UI (optional, nice-to-have) |

---

## 1. Install kubectl

```bash
curl -LO "https://dl.k8s.io/release/$(curl -L -s https://dl.k8s.io/release/stable.txt)/bin/linux/amd64/kubectl"
chmod +x kubectl
sudo mv kubectl /usr/local/bin/
kubectl version --client
```

Expected:
```
Client Version: v1.31.0
Kustomize Version: v5.4.0
```

> **Pro tip:** Add alias `k` for kubectl + auto-completion:
> ```bash
> echo 'source <(kubectl completion bash)' >> ~/.bashrc
> echo 'alias k=kubectl' >> ~/.bashrc
> echo 'complete -o default -F __start_kubectl k' >> ~/.bashrc
> source ~/.bashrc
> ```
> Now type `k get pods -A` instead of `kubectl get pods -A`.  

---

## 2. Install kind

**Kin**d = **K**ubernetes **in** **D**ocker. Runs clusters as containers.

```bash
curl -Lo kind https://kind.sigs.k8s.io/dl/latest/kind-linux-amd64
chmod +x kind
sudo mv kind /usr/local/bin/
kind version
```

Expected:
```
kind v0.24.0 go1.22.2 linux/amd64
```

---

## 3. Docker Desktop (Windows + WSL2)

Check Docker inside WSL:
```bash
docker ps
```

| Problem | Fix |
|---------|-----|
| `command not found` | Docker → Settings → WSL Integration → Enable your distro |
| `Cannot connect to daemon` | Start Docker Desktop, then `wsl --shutdown` in PowerShell, reopen Ubuntu |

---

## 4. Install k9s (Optional)

Terminal UI for K8s (like `htop` for clusters).

```bash
curl -Lo k9s.tar.gz https://github.com/derailed/k9s/releases/latest/download/k9s_Linux_amd64.tar.gz
tar -xvf k9s.tar.gz
sudo mv k9s /usr/local/bin/
```

---

## 🧪 Project 1: "My First Cluster"

Now, let's create your very first Kubernetes cluster using kind.

---

**Step 1: Create the cluster**

> Make sure Docker is running first.

```bash
kind create cluster --name my-first-cluster
```

This will pull a kind node image if not present, and create a Docker container named `my-first-cluster-control-plane`. Wait for the success message.

Expected:
```
Creating cluster "my-first-cluster" ...
 ✓ Ensuring node image 🖼
 ✓ Preparing nodes 📦
 ✓ Writing configuration 📜
 ✓ Starting control-plane 🕹️
 ✓ Installing CNI 🔌
 ✓ Installing StorageClass 💾
Set kubectl context to "kind-my-first-cluster"
```

---

**Step 2: Verify kubectl can talk to it**

```bash
kubectl get nodes
```

Expected:
```
NAME                             STATUS   ROLES           AGE   VERSION
my-first-cluster-control-plane   Ready    control-plane   30s   v1.27.x
```

This shows your single node is both control plane and worker (by default, kind uses the same node for both). The ROLES column says control-plane because it runs the control plane components; but it's also ready to run your workloads (pods).

---

**Step 3: Cluster info**

```bash
kubectl cluster-info
```

You'll see the Kubernetes control plane and CoreDNS addresses. These are internal services.

---

**Step 4: Understand the context**

```bash
kubectl config current-context
```

This should be `kind-my-first-cluster`. kubectl uses contexts to know which cluster to talk to.

---

### 📝 Must-Know Commands (So Far)

| Command | What it does |
|---------|-------------|
| `kubectl get nodes` | List all nodes in the cluster |
| `kubectl cluster-info` | See the cluster endpoints |
| `kubectl config view` | See your kubeconfig |
| `kubectl get pods -A` | List all pods in all namespaces (we'll use this soon) |
| `kubectl describe node <name>` | Detailed info about a node |
| `kind get clusters` | List kind clusters |
| `kind delete cluster --name my-first-cluster` | Destroy it (don't run now!) |

## 5. Bonus: Multi-Node Cluster

```bash
kind delete cluster --name my-first-cluster

cat <<EOF | kind create cluster --name k8-lab --config=-
kind: Cluster
apiVersion: kind.x-k8s.io/v1alpha4
nodes:
- role: control-plane
- role: worker
EOF

kubectl get nodes
```
```
NAME                 STATUS   ROLES           AGE   VERSION
k8-lab-control-plane Ready    control-plane   1m    v1.31.0
k8-lab-worker        Ready    <none>          1m    v1.31.0
```

Now you have a **real** control-plane + worker setup — like production.

---

## 6. Common Issues

| Problem | Likely Cause | Fix |
|---------|-------------|------|
| `kind` not found | Not in PATH | `sudo mv kind /usr/local/bin/` |
| `kind create cluster` hangs | Docker not running | Start Docker Desktop |
| `kubectl get nodes` → empty | Wrong context | `kubectl config use-context kind-...` |
| Docker permission denied | Not in docker group | `sudo usermod -aG docker $USER && newgrp docker` |
| Port conflict | Another cluster running | `kind delete clusters --all` |

---

## 7. Cleanup

```bash
kind delete cluster --name my-first-cluster
kind delete clusters --all      # delete everything
```

Kind clusters are cheap — break things, delete, recreate. That's the point.

---

## ✅ What I Learned

- **kubectl** = HTTP client for the API server
- **kind** = K8s inside Docker containers
- **Cluster** = control-plane + worker(s)
- **Control-plane** = API server, scheduler, controller manager, etcd
- **Worker node** runs your app pods
- **System pods** live in `kube-system` namespace
- `kubectl get nodes` — cluster members and health
- `kubectl cluster-info` — API server address
- `kind delete cluster` — instant cleanup

---

**Next → Phase 2: Your First Pod**